# Type Hints and Pydantic

## Python Type Hints: The Basics

### Annotating Variables and Functions

In [1]:
name: str = "large-model"
temperature: float = 0.7
max_output_tokens: int = 1024
is_streaming: bool = False

def format_prompt(system_message: str, user_query: str) -> str:
    return f"System: {system_message}\nUser: {user_query}"

temperature = "hot"  # No error, but mypy will flag it

### Collection Types


In [2]:
tags: list[str] = ["rag", "embeddings", "retrieval"]
token_counts: dict[str, int] = {"input": 150, "output": 320}
unique_models: set[str] = {"large-model", "small-model", "local-model"}
coordinates: tuple[float, float] = (37.7749, -122.4194)

from typing import List, Dict, Set, Tuple
tags: List[str] = ["rag", "embeddings", "retrieval"]

## Optional, Union, and None Types

### Optional
Optional[str] means str or None.

In [3]:
from typing import Optional

def call_llm(
    prompt: str,
    system_message: Optional[str] = None,
    temperature: Optional[float] = None,
) -> str:
    if system_message is not None:
        messages = [{"role": "system", "content": system_message}]
    else:
        messages = []
    messages.append({"role": "user", "content": prompt})

    return "response"

### Union
Optional[str] is shorthand for Union[str, None]

In [4]:
from typing import Union

def normalize_token_count(value: Union[int, str]) -> int:
    if isinstance(value, str):
        return int(value)
    return value

Starting with Python 3.10, you can use the pipe operator | instead of Union

In [6]:
def normalize_token_count(value: int | str) -> int:
    if isinstance(value, str):
        return int(value)
    return value

system_message: str | None = None

## Type Aliases, TypeVar, and Generics

### Type Aliases
A type alias gives a name to a type.

In [7]:
from typing import TypeAlias

def process1(messages: list[dict[str, str | list[dict[str, str]]]]) -> None:
    pass

Message: TypeAlias = dict[str, str | list[dict[str, str]]]

def process2(messages: list[Message]) -> None:
    pass

New way in Python 3.12+

In [8]:
type Message = dict[str, str | list[dict[str, str]]]

### TypeVar and Generics

In [9]:
from typing import TypeVar

T = TypeVar("T")

def first_element(items: list[T]) -> T:
    return items[0]

name: str = first_element(["large-model", "small-model"])
count: int = first_element([100, 200, 300])

You can also constrain a TypeVar to specific types

In [10]:
from typing import TypeVar

ModelId = TypeVar("ModelId", str, int)

def normalize_model_id(value: ModelId) -> ModelId:
    return value

### Generic Classes

In [11]:
from typing import Generic, TypeVar

T = TypeVar("T")

class BatchResult(Generic[T]):
    def __init__(self, items: list[T], total: int) -> None:
        self.items = items
        self.total = total

    def first(self) -> T:
        return self.items[0]

string_batch = BatchResult[str](items=["hello", "world"], total=2)
int_batch = BatchResult[int](items=[42, 99], total=2)

## Pydantic: Runtime Validation from Type Hints

### Defining a Model

A Pydantic model is a class that inherits from BaseModel

In [ ]:
%pip install pydantic
from pydantic import BaseModel

class LLMConfig(BaseModel):
    model: str
    temperature: float
    max_output_tokens: int
    stream: bool = False

config = LLMConfig(
    model="large-model",
    temperature=0.7,
    max_output_tokens=1024
)
print(config.model)        # "large-model"
print(config.temperature)  # 0.7
print(config.stream)       # False

large-model
0.7
False


### Automatic Validation and Coercion

In [20]:
config = LLMConfig(
    model="large-model",
    temperature="0.7",
    max_output_tokens="1024",
)
print(type(config.temperature))  # <class 'float'>

try:
    bad_config = LLMConfig(
        model="large-model",
        temperature="hot",
        max_output_tokens=1024,
    )
except Exception as e:
    print(e)

<class 'float'>
1 validation error for LLMConfig
temperature
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='hot', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


### Field() for Constraints and Metadata

In [21]:
from pydantic import BaseModel, Field

class LLMConfig(BaseModel):
    model: str = Field(description="The model identifier.")
    temperature: float = Field(
        default=0.7,
        ge=0.0,
        le=2.0,
        description="Sampling temperature. Higher = more creative."
    )
    max_output_tokens: int = Field(
        default=1024,
        ge=1,
        le=128000,
        description="Maximum output tokens in the response."
    )
    system_prompt: str = Field(
        default="You are a helpful assistant.",
        min_length=1,
        max_length=10000,
        description="The system prompt sent to the model."
    )

try:
    config = LLMConfig(model="large-model", temperature=5.0)
except Exception as e:
    print(e)

1 validation error for LLMConfig
temperature
  Input should be less than or equal to 2 [type=less_than_equal, input_value=5.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


### Nested Models

In [22]:
from pydantic import BaseModel, Field

class Message(BaseModel):
    role: str
    content: str

class TokenUsage(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

class Choice(BaseModel):
    index: int
    message: Message
    finish_reason: str

class LLMResponse(BaseModel):
    id: str
    model: str
    choices: list[Choice]
    usage: TokenUsage
    created: int

response = LLMResponse(
    id="chatcmpl-abc123",
    model="large-model",
    choices=[
        {
            "index": 0,
            "message": {"role": "assistant", "content": "Hello!"},
            "finish_reason": "stop",
        }
    ],
    usage={"prompt_tokens": 10, "completion_tokens": 5, "total_tokens": 15},
    created=1700000000,
)

print(response.choices[0].message.content)  # "Hello!"
print(response.usage.total_tokens)          # 15

Hello!
15


### Custom Validators

#### Field Validators
A @field_validator runs for one field. It receives the value and can return a cleaned value or raise an error

In [25]:
from pydantic import BaseModel, field_validator

class EmbeddingRequest(BaseModel):
    text: str
    model: str
    dimensions: int = 1536

    @field_validator("text")
    @classmethod
    def text_must_not_be_empty_or_whitespace(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("text must contain non-whitespace characters")
        return v.strip()

    @field_validator("model")
    @classmethod
    def model_must_be_supported(cls, v: str) -> str:
        supported = {"text-embedding-3-small", "text-embedding-3-large"}
        if v not in supported:
            raise ValueError(f"model must be one of {sorted(supported)}")
        return v

try:
    er = EmbeddingRequest(text="some text", model="text-embedding-3-medium")
except Exception as e:
    print(e)

1 validation error for EmbeddingRequest
model
  Value error, model must be one of ['text-embedding-3-large', 'text-embedding-3-small'] [type=value_error, input_value='text-embedding-3-medium', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


#### Model Validators
A @model_validator can run on the whole model. This is useful for cross-field validation

In [26]:
from typing import Self

from pydantic import BaseModel, model_validator

class GenerationConfig(BaseModel):
    temperature: float = 0.7
    top_p: float = 1.0
    top_k: int | None = None

    @model_validator(mode="after")
    def check_sampling_consistency(self) -> Self:
        if self.temperature == 0 and self.top_p < 1.0:
            raise ValueError(
                "When temperature is 0 (greedy), top_p should be 1.0 "
                "since sampling is disabled."
            )
        return self

### Serialization: Getting Data Out

#### model_dump() and model_dump_json()

In [27]:
from pydantic import BaseModel

class ChunkMetadata(BaseModel):
    source: str
    page: int
    chunk_index: int
    token_count: int

metadata = ChunkMetadata(
    source="architecture_patterns.pdf",
    page=42,
    chunk_index=7,
    token_count=256,
)

d = metadata.model_dump()
print(d)

j = metadata.model_dump_json()
print(j)

d = metadata.model_dump(exclude={"chunk_index"})
print(d)

{'source': 'architecture_patterns.pdf', 'page': 42, 'chunk_index': 7, 'token_count': 256}
{"source":"architecture_patterns.pdf","page":42,"chunk_index":7,"token_count":256}
{'source': 'architecture_patterns.pdf', 'page': 42, 'token_count': 256}


#### model_validate() and model_validate_json()

In [28]:
data = {"source": "notes.pdf", "page": 1, "chunk_index": 0, "token_count": 128}
metadata = ChunkMetadata.model_validate(data)

json_str = '{"source": "notes.pdf", "page": 1, "chunk_index": 0, "token_count": 128}'
metadata = ChunkMetadata.model_validate_json(json_str)

### Pydantic Settings for Configuration

In [29]:
%pip install pydantic-settings

  Using cached pydantic_settings-2.15.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached pydantic_settings-2.15.0-py3-none-any.whl (69 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, SecretStr

class AppSettings(BaseSettings):
    api_key: SecretStr

    default_model: str = "large-model"

    embedding_model: str = "text-embedding-3-small"

    chunk_size: int = Field(default=512, ge=100, le=8000)
    chunk_overlap: int = Field(default=50, ge=0)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)

    database_url: str = "sqlite:///app.db"
    qdrant_url: str = "http://localhost:6333"

    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        case_sensitive=False,
    )

settings = AppSettings()
print(settings.chunk_size)

api_key = settings.api_key.get_secret_value()

1024
